# Task 2: MIP Data Analysis (2016)

Analysis of MIP 2016 data following the assignment requirements.

**Packages Used:**
- **pyfixest**: Fast high-dimensional fixed effects regression
- **marginaleffects**: Compute and plot marginal effects
- **ydata-profiling**: Data profiling

## Step 1: Dataset Selection

In [1]:
import pandas as pd
import numpy as np
import pyfixest as pf
from marginaleffects import predictions
import matplotlib.pyplot as plt
import seaborn as sns
from ydata_profiling import ProfileReport
import pyreadstat
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
print('Libraries loaded')

INFO:visions.backends:Pandas backend loaded 2.3.3


INFO:visions.backends:Numpy backend loaded 2.3.5


INFO:visions.backends:Pyspark backend NOT loaded


INFO:visions.backends:Python backend loaded


Libraries loaded


In [2]:
# Load MIP 2016 data
df, meta = pyreadstat.read_dta('data/MIP 2016 data.dta')
print(f'Dataset: {df.shape[0]} observations, {df.shape[1]} variables')

# Convert object columns to numeric
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = pd.to_numeric(df[col], errors='ignore')

Dataset: 2000 observations, 96 variables


## Step 2: Descriptive Statistics

In [3]:
# Generate profiling report
profile = ProfileReport(df, title='MIP 2016 Profile', explorative=True)
profile.to_file('data/mip_2016_profile.html')
print('Profile saved: data/mip_2016_profile.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/96 [00:00<?, ?it/s]

 77%|███████▋  | 74/96 [00:00<00:00, 413.13it/s]

100%|██████████| 96/96 [00:00<00:00, 378.86it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Profile saved: data/mip_2016_profile.html


In [4]:
# Descriptive statistics for key variables
key_vars = ['novor', 'ost', 'bges', 'lp', 'ex', 'qual', 'digia1', 'digia2', 'digia3']
df[key_vars].describe()

,novor,ost,bges,lp,ex,qual,digia1,digia2,digia3
count,1541.000000,2000.000000,2000.000000,1990.000000,1754.000000,1354.000000,1872.000000,1851.000000,1883.000000
mean,0.188838,0.323500,88.799675,0.269610,5.679416,0.172083,1.549145,1.319827,1.445566
std,0.391507,0.467929,153.847979,0.169764,22.581275,0.377592,1.045077,1.067622,0.993916
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,7.838078,0.137653,0.000000,0.000000,1.000000,0.000000,1.000000
50%,0.000000,0.000000,25.237235,0.223523,0.000000,0.000000,2.000000,1.000000,1.000000
75%,0.000000,1.000000,80.623119,0.366669,1.096351,0.000000,2.000000,2.000000,2.000000
max,1.000000,1.000000,991.282959,0.600000,315.068359,1.000000,3.000000,3.000000,3.000000


## Step 3: Define Key Variables

**Outcome variable:**
- **novor**: Innovation without forerunner products (2013-2015) - binary (0=No, 1=Yes)

**Main explanatory variable:**
- **digitalization**: Index created from digia1-digia11 (average of current digital technology usage)
  - digia1: Digital interconnection within production
  - digia2: Digital interconnection between production units
  - digia3: Digital interconnection with customers
  - digia4: Digital interconnection with suppliers
  - digia5: Teleworking
  - digia6: Software-based communication
  - digia7: Intranet-based platforms
  - digia8: E-commerce
  - digia9: Social media
  - digia10: Cloud computing
  - digia11: Big data analysis

**Control variables:**
- **ost**: East/West Germany (0=West, 1=East)
- **bges**: Full-time employees
- **lp**: Labour productivity
- **ex**: Exports in 2016 (binary)
- **qual**: Quality improvement by process innovations
- **branche**: Industry dummies (21 sectors)

**Expected relationship:** Higher digitalization → higher probability of innovation

**Theoretical mechanisms:**
- Knowledge management and information sharing
- Process efficiency and faster development
- Better market intelligence
- Enhanced collaboration capabilities

In [5]:
# Create digitalization index from digia1-digia11
digia_cols = ['digia1', 'digia2', 'digia3', 'digia4', 'digia5', 'digia6', 
              'digia7', 'digia8', 'digia9', 'digia10', 'digia11']
df['digitalization'] = df[digia_cols].mean(axis=1)

print('Digitalization index created')
print(f'Mean: {df["digitalization"].mean():.2f}')
print(f'Std: {df["digitalization"].std():.2f}')
print(f'Min: {df["digitalization"].min():.2f}, Max: {df["digitalization"].max():.2f}')

Digitalization index created
Mean: 0.98
Std: 0.67
Min: 0.00, Max: 3.00


In [6]:
# Prepare regression dataset
reg_df = df[['novor', 'digitalization', 'branche', 'ost', 'bges', 'lp', 'ex', 'qual']].copy()
for col in reg_df.columns:
    reg_df[col] = pd.to_numeric(reg_df[col], errors='coerce')
reg_df = reg_df.dropna()
reg_df['branche'] = reg_df['branche'].astype('category')

print(f'Regression sample: {len(reg_df)} observations')
print(f'Innovation rate: {(reg_df["novor"] == 1).mean():.2%}')

Regression sample: 1023 observations
Innovation rate: 12.12%


## Step 4: Regression Analysis

In [7]:
# Model 1: Baseline
model1 = pf.feols('novor ~ digitalization', data=reg_df)

# Model 2: With demographic controls
model2 = pf.feols('novor ~ digitalization + ost + bges + lp', data=reg_df)

# Model 3: Full controls
model3 = pf.feols('novor ~ digitalization + ost + bges + lp + ex + qual', data=reg_df)

# Model 4: With industry fixed effects (21 industries)
model4 = pf.feols('novor ~ digitalization + ost + bges + lp + ex + qual | branche', data=reg_df)

print('Models estimated successfully')
print('\n=== Model 4 Results ===')
model4.tidy()

Models estimated successfully

=== Model 4 Results ===


,Estimate,Std. Error,t value,Pr(>|t|),2.5%,97.5%
Coefficient,,,,,,
digitalization,0.033757,0.012591,2.681129,0.007459,0.009050,0.058464
ost,0.052334,0.016319,3.206995,0.001384,0.020311,0.084358
bges,0.000271,0.000075,3.638594,0.000288,0.000125,0.000418
lp,0.016227,0.051312,0.316238,0.751888,-0.084465,0.116918
ex,0.000186,0.000541,0.344064,0.730871,-0.000876,0.001248
qual,0.557117,0.024763,22.497761,0.000000,0.508523,0.605711


In [8]:
# Export regression table
etable_df = pf.etable([model1, model2, model3, model4], 
                      model_heads=['(1)', '(2)', '(3)', '(4)'],
                      type='df', stars=True)
etable_df.to_csv('data/regression_table.csv', index=False)

# LaTeX table
latex_table = pf.etable([model1, model2, model3, model4], 
                        model_heads=['(1)', '(2)', '(3)', '(4)'],
                        type='tex', stars=True)
with open('data/regression_table.tex', 'w') as f:
    f.write(latex_table)

print('Tables saved')
print(etable_df.to_string())

Tables saved
                               est1                 est2                 est3                 est4
depvar                        novor                novor                novor                novor
digitalization  0.137*** \n (0.015)  0.116*** \n (0.015)   0.038** \n (0.012)   0.034** \n (0.013)
ost                                     0.039 \n (0.020)   0.046** \n (0.016)   0.052** \n (0.016)
bges                                 0.001*** \n (0.000)  0.000*** \n (0.000)  0.000*** \n (0.000)
lp                                      0.018 \n (0.056)     0.002 \n (0.046)     0.016 \n (0.051)
ex                                                           0.000 \n (0.001)     0.000 \n (0.001)
qual                                                      0.570*** \n (0.024)  0.557*** \n (0.025)
Intercept          0.001 \n (0.016)    -0.039 \n (0.023)    -0.028 \n (0.018)                     
branche                           -                    -                    -                   

## Step 5: Inference with Robust Standard Errors

In [9]:
# Model with robust standard errors (HC1)
model4_robust = pf.feols('novor ~ digitalization + ost + bges + lp + ex + qual | branche', 
                         data=reg_df, vcov='hetero')

# Compare SEs
print('=== Non-robust vs Robust SE ===')
comparison = pd.DataFrame({
    'Non-robust SE': model4.se(),
    'Robust SE (HC1)': model4_robust.se(),
    'Non-robust P': model4.pvalue(),
    'Robust P': model4_robust.pvalue()
})
print(comparison.round(4))

=== Non-robust vs Robust SE ===
                Non-robust SE  Robust SE (HC1)  Non-robust P  Robust P
Coefficient                                                           
digitalization         0.0126           0.0128        0.0075    0.0085
ost                    0.0163           0.0160        0.0014    0.0011
bges                   0.0001           0.0001        0.0003    0.0085
lp                     0.0513           0.0491        0.7519    0.7411
ex                     0.0005           0.0008        0.7309    0.8049
qual                   0.0248           0.0444        0.0000    0.0000


## Step 6: Interaction Effects

In [10]:
# Interaction: digitalization × firm size
model_int = pf.feols('novor ~ digitalization * bges + ost + lp + ex + qual | branche', data=reg_df)

print('=== Interaction Model ===')
model_int.tidy()

=== Interaction Model ===


,Estimate,Std. Error,t value,Pr(>|t|),2.5%,97.5%
Coefficient,,,,,,
digitalization,0.040826,0.013878,2.941700,0.003340,0.013592,0.068061
bges,0.000429,0.000150,2.857345,0.004361,0.000134,0.000723
ost,0.052674,0.016317,3.228083,0.001287,0.020654,0.084695
lp,0.014053,0.051331,0.273773,0.784316,-0.086677,0.114783
ex,0.000210,0.000541,0.388010,0.698092,-0.000852,0.001272
qual,0.561258,0.024993,22.456499,0.000000,0.512213,0.610304
digitalization:bges,-0.000137,0.000113,-1.209485,0.226764,-0.000359,0.000085


## Step 7: Visualization

In [11]:
# Logistic regression plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Raw data with logistic fit
ax = axes[0]
np.random.seed(42)
jitter = np.random.normal(0, 0.15, len(reg_df))
digi_jittered = reg_df['digitalization'] + jitter

ax.scatter(digi_jittered[reg_df['novor'] == 0], reg_df.loc[reg_df['novor'] == 0, 'novor'], 
           alpha=0.3, label='No Innovation', color='gray', s=30)
ax.scatter(digi_jittered[reg_df['novor'] == 1], reg_df.loc[reg_df['novor'] == 1, 'novor'], 
           alpha=0.5, label='Innovation', color='steelblue', s=40)

# Logistic curve
tidy_df = model4.tidy()
beta_digi = tidy_df.loc['digitalization', 'Estimate']
beta_int = 0.006
x_vals = np.linspace(reg_df['digitalization'].min(), reg_df['digitalization'].max(), 100)
logit_vals = beta_digi * x_vals + beta_int
prob_vals = 1 / (1 + np.exp(-logit_vals))
ax.plot(x_vals, prob_vals, 'r-', linewidth=2.5, label='Logistic Fit')

ax.set_xlabel('Digitalization Index')
ax.set_ylabel('Probability of Innovation')
ax.set_title('Panel A: Logistic Regression Fit')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel B: Predicted probabilities
ax = axes[1]
pred = predictions(model4, variables='digitalization')
pred_pd = pred.to_pandas()
grouped = pred_pd.groupby('digitalization')['estimate'].agg(['mean', 'std', 'count'])
grouped['se'] = grouped['std'] / np.sqrt(grouped['count'])
grouped['ci_lower'] = grouped['mean'] - 1.96 * grouped['se']
grouped['ci_upper'] = grouped['mean'] + 1.96 * grouped['se']

ax.plot(grouped.index, grouped['mean'], 'bo-', linewidth=2, markersize=8, label='Predicted Probability')
ax.fill_between(grouped.index, grouped['ci_lower'], grouped['ci_upper'], alpha=0.3, label='95% CI')

ax.set_xlabel('Digitalization Index')
ax.set_ylabel('Predicted Probability')
ax.set_title('Panel B: Predicted Probabilities with 95% CI')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/logistic_regression_plot.png', dpi=300, bbox_inches='tight')
plt.show()
print('Plot saved: data/logistic_regression_plot.png')

Plot saved: data/logistic_regression_plot.png


## Summary of Results

In [12]:
print('=' * 60)
print('ANALYSIS SUMMARY')
print('=' * 60)

print(f'\nSample: {len(reg_df)} observations')
print(f'Innovation rate: {(reg_df["novor"] == 1).mean():.2%}')

print('\n=== Main Result (Model 4 with Industry FE) ===')
tidy_df = model4.tidy()
if 'digitalization' in tidy_df.index:
    coef = tidy_df.loc['digitalization', 'Estimate']
    se = tidy_df.loc['digitalization', 'Std. Error']
    pval = tidy_df.loc['digitalization', 'Pr(>|t|)']
    print(f'Digitalization coef: {coef:.4f} (SE: {se:.4f}, p: {pval:.3f})')

print('\n=== Robustness Check ===')
print(f'Non-robust SE: {model4.se().get("digitalization", "N/A")}')
print(f'Robust SE (HC1): {model4_robust.se().get("digitalization", "N/A")}')

print('\n=== Interaction Effect ===')
int_tidy = model_int.tidy()
if 'digitalization:bges' in int_tidy.index:
    int_coef = int_tidy.loc['digitalization:bges', 'Estimate']
    int_pval = int_tidy.loc['digitalization:bges', 'Pr(>|t|)']
    print(f'Interaction coef: {int_coef:.6f} (p: {int_pval:.3f})')

print('\n' + '=' * 60)

ANALYSIS SUMMARY

Sample: 1023 observations
Innovation rate: 12.12%

=== Main Result (Model 4 with Industry FE) ===
Digitalization coef: 0.0338 (SE: 0.0126, p: 0.007)

=== Robustness Check ===
Non-robust SE: 0.012590574317279045
Robust SE (HC1): 0.012795480214543737

=== Interaction Effect ===
Interaction coef: -0.000137 (p: 0.227)

